# PS2 Image Catalog — Jupyter port

Per-image OCR + summary for each claim folder, plus a whole-claim summary fused from those records. This is the Python/Jupyter port of `ps2_image_catalog.ts` + `ps3_claim_summary.ts`.

**Walks every claim file** (PDF expanded page-by-page, JPGs as-is) and asks a vision-language model to describe each one. For every image / page the model returns:

- `image_type` — X-ray / IVP / KUB / USG / CT / MRI / Coronary Angiogram / Typed report / Stamp / …
- `body_region` — Abdomen / Chest / Coronary tree / etc.
- `modality_view` — PA / Lateral / RAO-Caudal / Transverse / N/A
- `stage_of_care` — pre / intra / post / uncertain
- `ocr` — `{ language, verbatim, english }` (all 12 Indic languages handled)
- `summary`, `key_observations`, `dates_found`, `image_quality`

After the per-image catalog is written, a second text-only call fuses the records into a whole-claim summary (patient, hospital, procedure, key findings, completeness, gaps).

**Two providers, switch via env var `PROVIDER`:**
- `fireworks` (default) — Fireworks AI; needs `FIREWORKS_API_KEY` in `.env`. Defaults to your `qwen3-vl-8b-instruct` deployment.
- `ollama` — opt-in: `PROVIDER=ollama`. Uses local Ollama at `http://localhost:11434/api/chat`, model `qwen3-vl:8b-instruct`.

**Output** — written next to the source PDFs/images inside the claim folder:
```
Claims/<PACKAGE>/<CLAIM>/image_catalog.json
Claims/<PACKAGE>/<CLAIM>/image_catalog.md
Claims/<PACKAGE>/<CLAIM>/claim_summary.md
```

Make sure `pyproject.toml` / `uv` deps are installed: `pymupdf`, `pillow`, `requests`, `json-repair`, `python-dotenv`.

In [1]:
import os, io, json, base64, time, sys, re
from pathlib import Path

import fitz  # PyMuPDF
from PIL import Image, ImageOps
import requests
from json_repair import repair_json
from dotenv import load_dotenv

load_dotenv()  # picks up OLLAMA_API_KEY / FIREWORKS_API_KEY / etc.

True

## Config

Override any of these at the shell or in a cell before running:
```python
os.environ['PROVIDER'] = 'fireworks'
os.environ['MODEL']    = 'accounts/.../deployments/<id>'
```

In [2]:
PROVIDER = os.environ.get('PROVIDER', 'fireworks')  # 'fireworks' (default) or 'ollama'

OLLAMA_URL        = os.environ.get('OLLAMA_URL',    'http://localhost:11434/api/chat')
FIREWORKS_URL     = os.environ.get('FIREWORKS_URL', 'https://api.fireworks.ai/inference/v1/chat/completions')
FIREWORKS_API_KEY = os.environ.get('FIREWORKS_API_KEY')

DEFAULT_MODELS = {
    'ollama':    'qwen3-vl:8b-instruct',
    'fireworks': 'accounts/ajeya-rao-k-eckusf6m/deployments/euufjyfd',  # qwen3-vl-8b-instruct
}
MODEL = os.environ.get('MODEL', DEFAULT_MODELS[PROVIDER])

if PROVIDER == 'fireworks' and not FIREWORKS_API_KEY:
    raise RuntimeError('PROVIDER=fireworks but FIREWORKS_API_KEY is not set in .env')

# Find the project root by looking for the Claims/ sibling folder. This lets
# the notebook run whether the cwd is the project root (Jupyter Lab launched
# from there) OR the code/ folder (VS Code's per-notebook cwd).
def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(4):
        if (p / 'Claims').is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
CLAIMS_ROOT  = PROJECT_ROOT / 'Claims'

MAX_EDGE    = 1280       # OCR-friendly resolution
JPEG_Q      = 90
PDF_DPI     = 220        # ~220 DPI, readable handwriting / Indic glyphs
NUM_PREDICT = 4096
TEMPERATURE = 0.05
TOP_K       = 30
REPEAT_PEN  = 1.0
SUPPORTED_IMG = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}

print(f'Provider:     {PROVIDER}')
print(f'Endpoint:     {FIREWORKS_URL if PROVIDER == "fireworks" else OLLAMA_URL}')
print(f'Model:        {MODEL}')
print(f'Project root: {PROJECT_ROOT}')

Provider:     fireworks
Endpoint:     https://api.fireworks.ai/inference/v1/chat/completions
Model:        accounts/ajeya-rao-k-eckusf6m/deployments/euufjyfd
Project root: G:\PROJECTS\medical_reports_summary


## System prompt

Verbatim port of the prompt in `ps2_image_catalog.ts`. Keep it identical so both pipelines emit the same JSON schema and reviewer-style language.

In [3]:
SYSTEM = r'''You are a medical document OCR + description assistant for India's
AB PM-JAY claims dataset. You receive ONE image at a time. The image may be:
  - A medical scan (X-ray, IVP, KUB, USG, CT, MRI, coronary angiogram still)
  - A typed clinical / radiology report (English or Indic-language)
  - A handwritten doctor's note or prescription
  - A doctor's stamp, signature block, hospital letterhead
  - A photograph of a cath-lab monitor

ROLE — STRICTLY DESCRIPTIVE
- Do not diagnose. Do not approve / reject. Do not recommend treatment.
- Describe what you SEE. Cite uncertainty when a feature is unclear.

OCR + LANGUAGE
- Transcribe every legible line VERBATIM in its original script for the
  "ocr.verbatim" field.
- Detect the language for "ocr.language": one of English, Hindi, Tamil, Telugu,
  Bengali, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Odia, Urdu,
  Assamese, mixed, or none (if no readable text).
- Translate the verbatim text into English for "ocr.english" (omit fillers,
  preserve clinical terms, drug names, doses, dates).
- When a date is in DD-MM-YYYY / DD.MM.YYYY / YYYY-MM-DD / Indic numerals,
  normalise to DD/MM/YYYY in "dates_found".

IMAGE_TYPE (pick ONE)
  "Chest X-Ray"   "Abdominal X-Ray (KUB)"   "IVP"   "Urethrogram"
  "Ultrasound"    "CT"   "MRI"   "Coronary Angiogram"
  "Typed report"  "Handwritten note"   "Stamp/Signature"
  "Hospital letterhead"   "Patient photo"   "Other"

For "Coronary Angiogram" specifically: cath-lab fluoroscopy frames have a
dark background, grey vessel structures, and on-screen UI overlays such as
"AXIOM-Artis", "RAO/LAO/CRAN/CAUD N°", "EE %", "DDO %", "WC", "WW", "CARD f/s",
or "73/F"-style patient header. ALL such monitor screenshots are
"Coronary Angiogram" — even when guidewires, balloons, or stents are present.

BODY_REGION (free text, anatomically precise)
  e.g. "Chest (PA)", "Abdomen — pelvicalyceal system", "Left kidney + ureter",
       "Coronary tree — left system", "Hepatobiliary system",
       "N/A (text document)"

MODALITY_VIEW (free text)
  e.g. "PA", "Lateral", "Oblique", "RAO 30° / Caudal 20°", "Transverse",
       "Sagittal", "Long-axis", "N/A"

OUTPUT — exactly ONE JSON object, no markdown, no code fence, no prose:

{
  "image_type":     "<from list above>",
  "body_region":    "<string or 'N/A'>",
  "modality_view":  "<string or 'N/A'>",
  "stage_of_care":  "pre-procedure" | "intra-procedure" | "post-procedure" | "uncertain" | "n/a",
  "stage_evidence": "<short clue>",
  "ocr": {
    "language":  "<English | Hindi | Tamil | Telugu | Bengali | Marathi | Gujarati | Kannada | Malayalam | Punjabi | Odia | Urdu | Assamese | mixed | none>",
    "verbatim":  "<every legible line in original script — newline-separated>",
    "english":   "<English translation; same as verbatim if already English>"
  },
  "summary":          "<1-3 sentences: what this image shows>",
  "key_observations": ["<bullet 1>", "<bullet 2>", "<bullet 3>"],
  "dates_found":      ["<DD/MM/YYYY>", "..."],
  "image_quality":    {
    "rating":      "good" | "fair" | "poor",
    "limitations": ["<e.g. partial view, low contrast, photograph of monitor, glare, blur>"]
  }
}

FORBIDDEN
- Diagnosing.
- Inventing text not visible in the image.
- Approving / rejecting claims.
- Markdown, code fences, or prose around the JSON.'''


## Image preprocessing — PDF→JPEG, EXIF rotate, resize, base64

PyMuPDF (`fitz`) replaces `pdf-to-img`; Pillow replaces `sharp`. Same effective behaviour: each page becomes a 220 DPI render, then the longest edge is capped at `MAX_EDGE` and re-encoded as quality-90 JPEG.

In [4]:
def pdf_to_pil_images(pdf_path: Path) -> list[Image.Image]:
    """Render every page of a PDF to a PIL Image at PDF_DPI."""
    doc = fitz.open(pdf_path)
    pages = []
    try:
        for page in doc:
            pix = page.get_pixmap(dpi=PDF_DPI, alpha=False)
            pages.append(Image.frombytes('RGB', (pix.width, pix.height), pix.samples))
    finally:
        doc.close()
    return pages

def optimize_image(img: Image.Image) -> bytes:
    """Honour EXIF rotation, downscale to MAX_EDGE on longest side, JPEG-encode."""
    img = ImageOps.exif_transpose(img)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.thumbnail((MAX_EDGE, MAX_EDGE), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=JPEG_Q, optimize=True)
    return buf.getvalue()

def image_to_b64(img: Image.Image) -> str:
    return base64.b64encode(optimize_image(img)).decode('ascii')

## Model call — provider-agnostic

Both providers share the same prompt and a normalised return shape `{content, prompt_eval_count, eval_count}`. Wire format differs:

- **Ollama** — image goes on the message as `images: [base64]`; raw base64 only, no data URL.
- **Fireworks (OpenAI-compat)** — image goes inside `content[]` as `{type: 'image_url', image_url: {url: 'data:image/jpeg;base64,…'}}`.

In [5]:
def user_instruction(label: str) -> str:
    return (
        f'IMAGE: {label}\n\n'
        'Produce the JSON object described in the system prompt. '
        'OCR every legible line verbatim. Translate non-English text to English. '
        'Describe what is visible — body region, modality, observations. '
        'ONE JSON object. NO PROSE.'
    )

def call_ollama(image_b64: str, label: str) -> dict:
    r = requests.post(OLLAMA_URL, json={
        'model':   MODEL,
        'stream':  False,
        'format':  'json',
        'messages': [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user',   'content': user_instruction(label), 'images': [image_b64]},
        ],
        'options': {
            'temperature':    TEMPERATURE,
            'top_p':          1,
            'top_k':          TOP_K,
            'repeat_penalty': REPEAT_PEN,
            'num_predict':    NUM_PREDICT,
        },
    }, timeout=300)
    r.raise_for_status()
    j = r.json()
    return {
        'content':           j.get('message', {}).get('content', ''),
        'prompt_eval_count': j.get('prompt_eval_count'),
        'eval_count':        j.get('eval_count'),
    }

def call_fireworks(image_b64: str, label: str) -> dict:
    r = requests.post(FIREWORKS_URL,
        headers={
            'Authorization': f'Bearer {FIREWORKS_API_KEY}',
            'Content-Type':  'application/json',
        },
        json={
            'model': MODEL,
            'messages': [
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user',   'content': [
                    {'type': 'text',      'text': user_instruction(label)},
                    {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
                ]},
            ],
            'response_format': {'type': 'json_object'},
            'temperature':     TEMPERATURE,
            'top_p':           1,
            'top_k':           TOP_K,
            'max_tokens':      NUM_PREDICT,
        },
        timeout=300,
    )
    r.raise_for_status()
    j = r.json()
    usage = j.get('usage', {}) or {}
    return {
        'content':           j['choices'][0]['message']['content'],
        'prompt_eval_count': usage.get('prompt_tokens'),
        'eval_count':        usage.get('completion_tokens'),
    }

def call_model(image_b64: str, label: str, retries: int = 3) -> dict:
    last = None
    for attempt in range(retries):
        try:
            return (call_fireworks if PROVIDER == 'fireworks' else call_ollama)(image_b64, label)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = 2 ** attempt + 1
                print(f'    ⚠  HTTP {code}; retry in {wait}s')
                time.sleep(wait); last = e; continue
            raise
    raise last  # type: ignore[misc]

## Robust JSON parse

Strips code fences, slices to the outermost `{...}`, falls back to `json_repair` if the model emits anything malformed.

In [6]:
def parse_json_obj(text: str) -> dict:
    s = text.strip()
    m = re.match(r'```(?:json)?\s*([\s\S]*?)\s*```', s, re.IGNORECASE)
    if m:
        s = m.group(1).strip()
    start, end = s.find('{'), s.rfind('}')
    if start != -1 and end > start:
        s = s[start:end+1]
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return json.loads(repair_json(s))

## Walk a claim folder → list of (source, page, total, PIL image)

In [7]:
def list_images_in_claim(claim_dir: Path) -> list[tuple[str, int | None, int | None, Image.Image]]:
    files = sorted([
        f for f in claim_dir.iterdir()
        if f.is_file() and (f.suffix.lower() in SUPPORTED_IMG or f.suffix.lower() == '.pdf')
    ])
    out: list[tuple[str, int | None, int | None, Image.Image]] = []
    for f in files:
        if f.suffix.lower() == '.pdf':
            pages = pdf_to_pil_images(f)
            for i, img in enumerate(pages):
                out.append((f.name, i + 1, len(pages), img))
        else:
            img = Image.open(f); img.load()
            out.append((f.name, None, None, img))
    return out

## Markdown summary renderer

Identical layout to the Bun version: an overview table at the top, then a per-image deep section with summary, key observations, dates, and a collapsed verbatim OCR block.

In [8]:
def render_markdown(claim_id: str, package_folder: str, records: list[dict]) -> str:
    L: list[str] = []
    L.append(f'# Image Catalog — {claim_id}'); L.append('')
    L.append(f'**Package:** {package_folder}  ·  **Model:** {MODEL}  ·  **Images:** {len(records)}'); L.append('')

    L.append('## Overview'); L.append('')
    L.append('| # | Source | Page | Type | Body region | Stage | Quality |')
    L.append('|---|--------|------|------|-------------|-------|---------|')
    for r in records:
        if 'error' in r:
            L.append(f"| {r['image_index']} | {r['source']} | {r.get('page') or '—'} | _error_ | — | — | — |")
            continue
        pg = f"{r['page']}/{r['pages_total']}" if r.get('page') else '—'
        rating = (r.get('image_quality') or {}).get('rating', '?')
        L.append(
            f"| {r['image_index']} | {r['source']} | {pg} | {r.get('image_type','?')} | "
            f"{r.get('body_region','?')} | {r.get('stage_of_care','?')} | {rating} |"
        )
    L.append('')

    for r in records:
        pg = f" — page {r['page']}/{r['pages_total']}" if r.get('page') else ''
        L.append(f"## [{r['image_index']}] {r['source']}{pg}"); L.append('')
        if 'error' in r:
            L.append(f"> **Error:** {r['error']}"); L.append(''); continue

        meta: list[str] = []
        if r.get('image_type'):  meta.append(f"**Type:** {r['image_type']}")
        if r.get('body_region'): meta.append(f"**Body region:** {r['body_region']}")
        if r.get('modality_view') and r['modality_view'] != 'N/A':
            meta.append(f"**View:** {r['modality_view']}")
        if r.get('stage_of_care'):
            ev = f" ({r['stage_evidence']})" if r.get('stage_evidence') else ''
            meta.append(f"**Stage:** {r['stage_of_care']}{ev}")
        q = r.get('image_quality') or {}
        if q.get('rating'):
            lim = [x for x in (q.get('limitations') or []) if x]
            extra = f" — {', '.join(lim)}" if lim else ''
            meta.append(f"**Quality:** {q['rating']}{extra}")
        if meta: L.append('  ·  '.join(meta)); L.append('')

        if r.get('summary'): L.append(f"**Summary.** {r['summary']}"); L.append('')

        kobs = r.get('key_observations') or []
        if kobs:
            L.append('**Key observations:**')
            for o in kobs: L.append(f'- {o}')
            L.append('')

        dates = r.get('dates_found') or []
        if dates: L.append(f"**Dates found:** {', '.join(dates)}"); L.append('')

        ocr = r.get('ocr') or {}
        verb = (ocr.get('verbatim') or '').strip()
        eng  = (ocr.get('english') or '').strip()
        if verb or eng:
            L.append(f"**OCR** _(language: {ocr.get('language','?')})_")
            if verb:
                L.append(''); L.append('<details><summary>Verbatim (original script)</summary>'); L.append('')
                L.append('```'); L.append(verb); L.append('```'); L.append('')
                L.append('</details>')
            if eng and eng != verb:
                L.append(''); L.append('**English:**'); L.append('')
                L.append('```'); L.append(eng); L.append('```')
            L.append('')
        L.append('---'); L.append('')
    return '\n'.join(L)

## Claim summary — dashboard-ready whole-claim synthesis

After the per-image catalog is written, fuse every record into a single `claim_summary.{json,md}` matching the reviewer dashboard schema (header bar, status banner, AI clinical findings, multi-image timeline, report NLP extraction, finding correlation, inconsistency detection, STG alignment, radiology timeline, plus reference patient/hospital/encounter/inventory detail).

Two parallel model calls plus several deterministic derivations in code:

- **Two prompts** (`SUMMARY_SYSTEM_DASHBOARD`, `SUMMARY_SYSTEM_REFERENCE`) → smaller schemas → fewer null fields than a single big call.
- **Code-side derivations** for sections the model is unreliable at: `finding_correlation` is built from `status.key_findings`, `multi_image_analysis` and `radiology_timeline` are built from per-image dates/types, `completeness` is built from per-image `stage_of_care` / `image_type`.
- **Strict null handling**: missing fields become literal `null` (not `"unknown"` / `"N/A"`); unknown booleans become `null`; percentages are integers; dates are DD/MM/YYYY.
- **Defensive plumbing**: `frequency_penalty: 0.4` to suppress n-gram loops on repetitive OCR junk; key normalisation maps frequent model typos (`Clinical_narrative` / `kay_findings` / `is_*_present`) back to canonical names; `"unknown"` strings get coerced to `null` in a deep walk.

Output: `Claims/<PACKAGE>/<CLAIM>/claim_summary.json` (frontend dashboard data) + `claim_summary.md` (human-readable preview).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

# ── Two prompts: dashboard schema + reference detail ────────────────────
SUMMARY_SYSTEM_DASHBOARD = r"""You are a medical-claim summarisation assistant for India's
AB PM-JAY claims dataset. You receive a JSON list of per-image OCR + description
records produced by a vision model for ONE claim folder. Fuse those records
into a SINGLE whole-claim summary in the schema below — a frontend reviewer
dashboard renders the result directly.

ROLE — STRICTLY DESCRIPTIVE
- Do not diagnose. Do not approve / reject. Do not recommend treatment.
- Describe what the dossier shows, gaps included.

NULL HANDLING — VERY IMPORTANT
- Use the JSON literal null (NOT the string "null", NOT "unknown", NOT "N/A")
  for any field whose value cannot be derived from the records.
- Booleans must be true / false literals. If unknown → null.
- Percentages are integers 0-100 with no % sign. If unknown → null.
- Dates: DD/MM/YYYY string only. If unknown → null.
- Empty array [] = "checked, none found". null = "cannot determine".
- Never invent identifiers, dates, or findings.

INPUTS YOU CAN USE
- Per-record fields: source, page, image_type, body_region, modality_view,
  stage_of_care, summary, key_observations, dates_found, image_quality, ocr.
- Treat ocr.english as the primary text source for "report" content.
- Image observations describe what the IMAGES show.
- "Report" content = OCR'd typed reports / handwritten notes inside the dossier.

OUTPUT — exactly ONE JSON object matching this schema. No markdown, no prose.

{
  "header": {
    "claim_id":     "<copy from input>",
    "patient_name": "<string|null>",
    "modality":     "<dominant modality e.g. 'X-Ray'|'CT'|'MRI'|'Ultrasound'|'Coronary Angiogram'|'Mixed'|null>",
    "body_part":    "<dominant body region e.g. 'Tibia'|'Coronary tree'|'Abdomen'|null>",
    "study_date":   "<earliest study date DD/MM/YYYY|null>",
    "reviewer":     null
  },
  "status": {
    "consistency":         "consistent|partial|mismatch|null",
    "confidence_pct":      <int 0-100|null>,
    "clinical_risk_score": "Low|Medium|High|null",
    "key_findings": [
      { "finding": "<short label>", "ai_detected": <bool|null>, "report_mentioned": <bool|null>, "note": "<string|null>" }
    ]
  },
  "scan_viewer": {
    "primary_image_source": "<best representative filename|null>",
    "detected_regions": [
      { "label": "<e.g. 'Fracture zone'>", "image_source": "<filename>", "page": <int|null> }
    ],
    "ai_overlays_available": false
  },
  "ai_clinical_findings": {
    "fracture":           { "present": <bool|null>, "confidence_pct": <int|null>, "evidence": "<string|null>" },
    "fluid_accumulation": { "present": <bool|null>, "confidence_pct": <int|null>, "evidence": "<string|null>" },
    "tumor_mass":         { "present": <bool|null>, "confidence_pct": <int|null>, "evidence": "<string|null>" },
    "infiltration":       { "severity": "None|Mild|Moderate|Severe|null", "confidence_pct": <int|null>, "evidence": "<string|null>" },
    "image_quality":      "good|fair|poor|null"
  },
  "report_nlp_extraction": {
    "reported_diagnosis":    "<string|null>",
    "reported_severity":     "Minor|Moderate|Severe|null",
    "reported_findings":     ["<string>"],
    "extraction_confidence": "High|Medium|Low|null"
  },
  "inconsistency_detection": {
    "possible_exaggerations": ["<string>"],
    "underreported_findings": ["<string>"],
    "hidden_findings":        ["<string>"]
  },
  "stg_alignment": {
    "claimed_package":          "<string|null>",
    "evidence_required":        [ { "item": "<string>", "present": <bool|null> } ],
    "stg_compliance_score_pct": <int|null>
  }
}

GUIDANCE FOR SPECIFIC SECTIONS

status.consistency: "consistent" if image findings and report broadly agree;
"partial" if some mismatch; "mismatch" if major conflicts.

ai_clinical_findings: report on what THE IMAGES (not the report) show.
.evidence should cite the image source filename(s).

report_nlp_extraction: report on what the WRITTEN REPORTS / NOTES (OCR) say.
Independent of image findings.

inconsistency_detection:
  possible_exaggerations = report claims things images don't show.
  underreported_findings = images show things report didn't mention.
  hidden_findings        = anything obviously omitted from both.

stg_alignment: PMJAY package codes look like "MC011A", "SU007A". claimed_package
is the input package. evidence_required is a short generic checklist for that
procedure type. Mark .present per the dossier.

FORBIDDEN
- Diagnosing.
- Approving / rejecting the claim.
- Inventing names, IDs, or dates that do not appear in the records.
- Returning the string "null" instead of the literal null.
- Markdown, code fences, or prose around the JSON."""

SUMMARY_SYSTEM_REFERENCE = r"""You are a medical-claim summarisation assistant for India's
AB PM-JAY claims dataset. You receive a JSON list of per-image OCR + description
records produced by a vision model for ONE claim folder. Extract patient,
hospital, encounter, image-inventory, narrative, completeness, and gap details.

ROLE — STRICTLY DESCRIPTIVE. No diagnoses. No claim approval / rejection.

NULL HANDLING
- Use literal null (not "unknown", not "N/A", not the string "null") for any
  field you cannot derive from the records.
- Booleans must be true/false literals. Unknown → null.
- Dates: DD/MM/YYYY only.
- Empty array [] = "checked, none found"; null = "cannot determine".
- Never invent.

OUTPUT — exactly ONE JSON object. No markdown. No prose.

{
  "patient": {
    "name":       "<string|null>",
    "age":        "<e.g. '54 years'|null>",
    "sex":        "Male|Female|null",
    "id_numbers": ["<MRN/UHID/Aadhaar etc.>"]
  },
  "hospital": {
    "name":     "<string|null>",
    "location": "<city/state|null>",
    "doctors":  ["<string with degrees if visible>"]
  },
  "encounter": {
    "date_range":        "<DD/MM/YYYY to DD/MM/YYYY|null>",
    "all_dates":         ["<every distinct DD/MM/YYYY>"],
    "primary_procedure": "<short phrase|null>",
    "package_code":      "<PMJAY scheme code|null>"
  },
  "image_inventory": {
    "total_images":   <int>,
    "by_type":        { "<image_type>": <count> },
    "stages_present": ["pre-procedure|intra-procedure|post-procedure|uncertain|n/a"],
    "languages_seen": ["English|Hindi|..."]
  },
  "clinical_narrative": "<2-4 sentence plain-English description|null>",
  "completeness": {
    "has_pre_procedure_imaging":   <bool|null>,
    "has_intra_procedure_imaging": <bool|null>,
    "has_post_procedure_imaging":  <bool|null>,
    "has_typed_report":            <bool|null>,
    "has_handwritten_notes":       <bool|null>,
    "has_signed_stamp":            <bool|null>,
    "notes":                       "<string|null>"
  },
  "concerns_or_gaps": ["<string>"]
}

FORBIDDEN
- Diagnosing / approving / rejecting.
- Inventing names, IDs, or dates.
- Markdown, code fences, or prose around the JSON."""

SUMMARY_TEMPERATURE  = 0.1
SUMMARY_NUM_PREDICT  = 16000
MAX_LIST_PER_RECORD  = 8
MAX_OCR_CHARS        = 1200

def _dedup_cap(arr, max_n=MAX_LIST_PER_RECORD):
    if not isinstance(arr, list): return []
    seen, out = set(), []
    for v in arr:
        s = str(v if v is not None else '').strip()
        if not s or s in seen: continue
        seen.add(s); out.append(s)
        if len(out) >= max_n: break
    return out

def _trim_text(s, max_n=MAX_OCR_CHARS):
    if not isinstance(s, str): return ''
    collapsed = re.sub(r'[ \t]+', ' ', s)
    collapsed = re.sub(r'\n{3,}', '\n\n', collapsed).strip()
    return collapsed if len(collapsed) <= max_n else collapsed[:max_n] + '…[truncated]'

def compact_catalog(catalog: dict) -> dict:
    """Strip _usage / stage_evidence and dedupe noisy fields so the summariser
    isn't primed to loop on repeated OCR junk."""
    records = []
    for r in (catalog.get('records') or []):
        if 'error' in r:
            records.append({'image_index': r.get('image_index'), 'source': r.get('source'),
                            'page': r.get('page'), 'error': r.get('error')})
            continue
        ocr = r.get('ocr') or {}
        records.append({
            'image_index':      r.get('image_index'),
            'source':           r.get('source'),
            'page':             r.get('page'),
            'pages_total':      r.get('pages_total'),
            'image_type':       r.get('image_type'),
            'body_region':      r.get('body_region'),
            'modality_view':    r.get('modality_view'),
            'stage_of_care':    r.get('stage_of_care'),
            'summary':          _trim_text(r.get('summary'), 600),
            'key_observations': _dedup_cap(r.get('key_observations')),
            'dates_found':      _dedup_cap(r.get('dates_found')),
            'ocr': {
                'language': ocr.get('language'),
                'verbatim': _trim_text(ocr.get('verbatim')),
                'english':  _trim_text(ocr.get('english')),
            },
            'image_quality':    r.get('image_quality'),
        })
    return {
        'claim_id': catalog.get('claim_id'),
        'package':  catalog.get('package'),
        'n_images': catalog.get('n_images') or len(records),
        'records':  records,
    }

# ── Provider call (text-only; system + user) ────────────────────────────
def _call_text_fireworks(system_text: str, user_text: str) -> str:
    r = requests.post(FIREWORKS_URL,
        headers={'Authorization': f'Bearer {FIREWORKS_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': MODEL,
            'messages': [
                {'role': 'system', 'content': system_text},
                {'role': 'user',   'content': user_text},
            ],
            'response_format':   {'type': 'json_object'},
            'temperature':       SUMMARY_TEMPERATURE,
            'max_tokens':        SUMMARY_NUM_PREDICT,
            'frequency_penalty': 0.4,
            'presence_penalty':  0.1,
        },
        timeout=600,
    )
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

def _call_text_ollama(system_text: str, user_text: str) -> str:
    r = requests.post(OLLAMA_URL, json={
        'model':  MODEL,
        'stream': False,
        'format': 'json',
        'messages': [
            {'role': 'system', 'content': system_text},
            {'role': 'user',   'content': user_text},
        ],
        'options': {'temperature': SUMMARY_TEMPERATURE, 'num_predict': SUMMARY_NUM_PREDICT},
    }, timeout=600)
    r.raise_for_status()
    return r.json().get('message', {}).get('content', '')

def call_text_model(system_text: str, user_text: str, retries: int = 5) -> str:
    last = None
    for attempt in range(retries):
        try:
            return (_call_text_fireworks if PROVIDER == 'fireworks' else _call_text_ollama)(system_text, user_text)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            body = e.response.text if e.response is not None else ''
            scaling_up = code == 503 and re.search(r'scaling.up|scaled.to.zero', body, re.I)
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = 30 * (attempt + 1) if scaling_up else (2 ** attempt + 1)
                tag = ' (deployment scaling up)' if scaling_up else ''
                print(f'    ⚠  HTTP {code}{tag}; retry in {wait}s')
                time.sleep(wait); last = e; continue
            raise
    raise last  # type: ignore[misc]

# ── Schema normalization (null-coercion + key drift mapping) ────────────
NULLISH_STRINGS = {'', 'unknown', 'n/a', 'na', 'none', 'null', '?', '—', '-'}

def _nz_str(v):
    if v is None: return None
    if not isinstance(v, str): return None if v == '' else str(v)
    s = v.strip()
    return None if s.lower() in NULLISH_STRINGS else s

def _nz_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return v != 0
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ('true', 'yes', 'y', '1', 'present'): return True
        if s in ('false', 'no', 'n', '0', 'absent'):  return False
    return None

def _nz_int(v):
    if isinstance(v, bool): return None
    if isinstance(v, (int, float)): return int(round(v))
    if isinstance(v, str):
        m = re.search(r'-?\d+(?:\.\d+)?', v)
        if m: return int(round(float(m.group(0))))
    return None

def _nz_arr(v):
    return v if isinstance(v, list) else None

def _nz_enum(v, allowed):
    s = _nz_str(v)
    if s is None: return None
    for a in allowed:
        if a.lower() == s.lower(): return a
    return None

def _deep_nullify(v):
    if v is None: return None
    if isinstance(v, str):
        return None if v.strip().lower() in NULLISH_STRINGS else v
    if isinstance(v, list):
        return [_deep_nullify(x) for x in v]
    if isinstance(v, dict):
        return {k: _deep_nullify(val) for k, val in v.items()}
    return v

def _pick_key(obj, *candidates):
    if not isinstance(obj, dict): return None
    for k in candidates:
        if k in obj: return obj[k]
    lower = {k.lower(): k for k in obj.keys()}
    for k in candidates:
        hit = lower.get(k.lower())
        if hit is not None: return obj[hit]
    return None

def _normalize_finding(f):
    if not isinstance(f, dict): f = {}
    return {
        'present':        _nz_bool(_pick_key(f, 'present', 'detected', 'is_present')),
        'confidence_pct': _nz_int(_pick_key(f, 'confidence_pct', 'confidence', 'confidence_percentage')),
        'evidence':       _nz_str(_pick_key(f, 'evidence', 'source', 'note')),
    }

def normalize_summary(raw: dict, package_folder: str, claim_id: str) -> dict:
    raw = _deep_nullify(raw or {})
    header     = _pick_key(raw, 'header') or {}
    status     = _pick_key(raw, 'status') or {}
    scan       = _pick_key(raw, 'scan_viewer', 'scanViewer') or {}
    ai_f       = _pick_key(raw, 'ai_clinical_findings', 'aiClinicalFindings', 'ai_findings') or {}
    nlp        = _pick_key(raw, 'report_nlp_extraction', 'reportNlpExtraction', 'report_extraction') or {}
    inc        = _pick_key(raw, 'inconsistency_detection', 'inconsistencyDetection') or {}
    stg        = _pick_key(raw, 'stg_alignment', 'stgAlignment') or {}
    patient    = _pick_key(raw, 'patient') or {}
    hospital   = _pick_key(raw, 'hospital') or {}
    encounter  = _pick_key(raw, 'encounter') or {}
    inv        = _pick_key(raw, 'image_inventory', 'imageInventory') or {}
    c          = _pick_key(raw, 'completeness') or {}

    return {
        'header': {
            'claim_id':     claim_id,
            'patient_name': _nz_str(_pick_key(header, 'patient_name', 'patientName', 'name')) or _nz_str(_pick_key(patient, 'name')),
            'modality':     _nz_str(_pick_key(header, 'modality')),
            'body_part':    _nz_str(_pick_key(header, 'body_part', 'bodyPart', 'body_region')),
            'study_date':   _nz_str(_pick_key(header, 'study_date', 'studyDate', 'date')),
            'reviewer':     None,
        },
        'status': {
            'consistency':         _nz_enum(_pick_key(status, 'consistency'), ['consistent', 'partial', 'mismatch']),
            'confidence_pct':      _nz_int(_pick_key(status, 'confidence_pct', 'confidence')),
            'clinical_risk_score': _nz_enum(_pick_key(status, 'clinical_risk_score', 'clinicalRiskScore', 'risk'), ['Low', 'Medium', 'High']),
            'key_findings': [
                {
                    'finding':          _nz_str(_pick_key(f, 'finding', 'label', 'name')),
                    'ai_detected':      _nz_bool(_pick_key(f, 'ai_detected', 'image_ai', 'ai')),
                    'report_mentioned': _nz_bool(_pick_key(f, 'report_mentioned', 'report', 'in_report')),
                    'note':             _nz_str(_pick_key(f, 'note', 'evidence', 'detail')),
                }
                for f in (_nz_arr(_pick_key(status, 'key_findings', 'keyFindings', 'findings')) or [])
            ],
        },
        'scan_viewer': {
            'primary_image_source': _nz_str(_pick_key(scan, 'primary_image_source', 'primary', 'source')),
            'detected_regions': [
                {
                    'label':        _nz_str(_pick_key(r, 'label', 'name')),
                    'image_source': _nz_str(_pick_key(r, 'image_source', 'source')),
                    'page':         _nz_int(_pick_key(r, 'page')),
                }
                for r in (_nz_arr(_pick_key(scan, 'detected_regions', 'regions')) or [])
            ],
            'ai_overlays_available': False,
        },
        'ai_clinical_findings': {
            'fracture':           _normalize_finding(_pick_key(ai_f, 'fracture')),
            'fluid_accumulation': _normalize_finding(_pick_key(ai_f, 'fluid_accumulation', 'fluidAccumulation', 'fluid')),
            'tumor_mass':         _normalize_finding(_pick_key(ai_f, 'tumor_mass', 'tumorMass', 'mass', 'tumor')),
            'infiltration': {
                'severity':       _nz_enum(_pick_key(_pick_key(ai_f, 'infiltration') or {}, 'severity'), ['None', 'Mild', 'Moderate', 'Severe']),
                'confidence_pct': _nz_int(_pick_key(_pick_key(ai_f, 'infiltration') or {}, 'confidence_pct', 'confidence')),
                'evidence':       _nz_str(_pick_key(_pick_key(ai_f, 'infiltration') or {}, 'evidence')),
            },
            'image_quality': _nz_enum(_pick_key(ai_f, 'image_quality', 'imageQuality'), ['good', 'fair', 'poor']),
        },
        'report_nlp_extraction': {
            'reported_diagnosis':    _nz_str(_pick_key(nlp, 'reported_diagnosis', 'diagnosis')),
            'reported_severity':     _nz_enum(_pick_key(nlp, 'reported_severity', 'severity'), ['Minor', 'Moderate', 'Severe']),
            'reported_findings':     [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(nlp, 'reported_findings', 'findings')) or [])] if x],
            'extraction_confidence': _nz_enum(_pick_key(nlp, 'extraction_confidence', 'confidence'), ['High', 'Medium', 'Low']),
        },
        'inconsistency_detection': {
            'possible_exaggerations': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(inc, 'possible_exaggerations', 'exaggerations')) or [])] if x],
            'underreported_findings': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(inc, 'underreported_findings', 'underreported')) or [])] if x],
            'hidden_findings':        [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(inc, 'hidden_findings', 'hidden')) or [])] if x],
        },
        'stg_alignment': {
            'claimed_package': _nz_str(_pick_key(stg, 'claimed_package', 'claimedPackage', 'package')) or package_folder,
            'evidence_required': [
                { 'item': _nz_str(_pick_key(e, 'item', 'label', 'name')), 'present': _nz_bool(_pick_key(e, 'present', 'satisfied')) }
                for e in (_nz_arr(_pick_key(stg, 'evidence_required', 'evidence', 'checklist')) or [])
            ],
            'stg_compliance_score_pct': _nz_int(_pick_key(stg, 'stg_compliance_score_pct', 'compliance_score', 'complianceScore')),
        },
        'patient': {
            'name':       _nz_str(_pick_key(patient, 'name')),
            'age':        _nz_str(_pick_key(patient, 'age')),
            'sex':        _nz_enum(_pick_key(patient, 'sex'), ['Male', 'Female']),
            'id_numbers': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(patient, 'id_numbers', 'ids')) or [])] if x],
        },
        'hospital': {
            'name':     _nz_str(_pick_key(hospital, 'name')),
            'location': _nz_str(_pick_key(hospital, 'location')),
            'doctors':  [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(hospital, 'doctors')) or [])] if x],
        },
        'encounter': {
            'date_range':        _nz_str(_pick_key(encounter, 'date_range', 'dateRange')),
            'all_dates':         [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(encounter, 'all_dates', 'allDates', 'dates')) or [])] if x],
            'primary_procedure': _nz_str(_pick_key(encounter, 'primary_procedure', 'procedure')),
            'package_code':      _nz_str(_pick_key(encounter, 'package_code', 'packageCode')),
        },
        'image_inventory': {
            'total_images':   _nz_int(_pick_key(inv, 'total_images', 'totalImages')),
            'by_type':        _pick_key(inv, 'by_type', 'byType') or {},
            'stages_present': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(inv, 'stages_present', 'stagesPresent')) or [])] if x],
            'languages_seen': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(inv, 'languages_seen', 'languagesSeen')) or [])] if x],
        },
        'clinical_narrative': _nz_str(_pick_key(raw, 'clinical_narrative', 'Clinical_narrative', 'clinicalNarrative', 'narrative')),
        'completeness': {
            'has_pre_procedure_imaging':   _nz_bool(_pick_key(c, 'has_pre_procedure_imaging',   'is_pre_procedure_imaging_present',   'pre_procedure_imaging')),
            'has_intra_procedure_imaging': _nz_bool(_pick_key(c, 'has_intra_procedure_imaging', 'is_intra_procedure_imaging_present', 'intra_procedure_imaging')),
            'has_post_procedure_imaging':  _nz_bool(_pick_key(c, 'has_post_procedure_imaging',  'is_post_procedure_imaging_present',  'post_procedure_imaging')),
            'has_typed_report':            _nz_bool(_pick_key(c, 'has_typed_report',            'is_typed_report_present',            'typed_report')),
            'has_handwritten_notes':       _nz_bool(_pick_key(c, 'has_handwritten_notes',       'is_handwritten_notes_present',       'handwritten_notes')),
            'has_signed_stamp':            _nz_bool(_pick_key(c, 'has_signed_stamp',            'is_signed_stamp_present',            'signed_stamp', 'has_stamp')),
            'notes':                       _nz_str(_pick_key(c, 'notes', 'note')),
        },
        'concerns_or_gaps': [x for x in [_nz_str(y) for y in (_nz_arr(_pick_key(raw, 'concerns_or_gaps', 'gaps_or_concerns', 'concerns', 'gaps')) or [])] if x],
    }

# ── Deterministic derivations from the per-image catalog ────────────────
def _parse_date(s):
    if not isinstance(s, str): return None
    m = re.search(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    if not m: return None
    try: return datetime(int(m.group(3)), int(m.group(2)), int(m.group(1)))
    except ValueError: return None

def _derive_finding_correlation(s):
    kf = (s.get('status') or {}).get('key_findings') or []
    rows = []
    for f in kf:
        ai, rp = f.get('ai_detected'), f.get('report_mentioned')
        match = (ai == rp) if (ai is not None and rp is not None) else None
        rows.append({'finding': f.get('finding'), 'image_ai': ai, 'report': rp, 'match': match})
    decided = [r for r in rows if r['match'] is not None]
    score = round(100 * sum(1 for r in decided if r['match']) / len(decided)) if decided else None
    return {'rows': rows, 'consistency_score_pct': score}

def _derive_multi_image_analysis(catalog):
    records = catalog.get('records') or []
    groups = {}
    for r in records:
        if 'error' in r: continue
        t = r.get('image_type') or 'Other'
        groups.setdefault(t, []).append(r)

    all_dates = [(_parse_date(d), d) for r in records for d in (r.get('dates_found') or []) if _parse_date(d)]
    min_date = min((d for d, _ in all_dates), default=None)

    entries = []
    for modality, rs in groups.items():
        ds = [(p, s) for r in rs for s in (r.get('dates_found') or []) for p in [_parse_date(s)] if p]
        ds.sort(key=lambda x: x[0])
        date = ds[0][1] if ds else None
        d_obj = ds[0][0] if ds else None
        day = ((d_obj - min_date).days + 1) if (d_obj and min_date) else None
        confirmed = any(
            r.get('stage_of_care') and str(r['stage_of_care']).lower() not in ('uncertain', 'n/a', 'none')
            for r in rs
        )
        finding = next((r.get('body_region') for r in rs if r.get('body_region')), None)
        entries.append({'modality': modality, 'day': day, 'date': date, 'finding': finding, 'confirmed': confirmed})
    return {'entries': entries, 'consistency_score_pct': None}

def _derive_radiology_timeline(catalog):
    records = catalog.get('records') or []
    pairs = {}
    min_date = None
    for r in records:
        if 'error' in r: continue
        modality = r.get('image_type') or 'Other'
        for ds in (r.get('dates_found') or []):
            d = _parse_date(ds)
            if not d: continue
            if min_date is None or d < min_date: min_date = d
            pairs.setdefault((ds, modality), {'date': ds, 'modality': modality, 'date_obj': d})
    events = sorted(pairs.values(), key=lambda p: p['date_obj'])
    out = [{'day': ((p['date_obj'] - min_date).days + 1) if min_date else None,
            'date': p['date'], 'event': p['modality']} for p in events]
    return {'events': out, 'logical': True if out else None}

def _derive_completeness(s, catalog):
    records = catalog.get('records') or []
    stages = {str(r.get('stage_of_care') or '').lower() for r in records}
    types  = {str(r.get('image_type') or '').lower()    for r in records}
    has_stage = (lambda needle: None if not records else any(needle in s for s in stages))
    has_type  = (lambda needle: None if not records else any(needle in t for t in types))
    c = s.get('completeness') or {}
    merge = lambda model_v, derived: model_v if isinstance(model_v, bool) else derived
    return {
        'has_pre_procedure_imaging':   merge(c.get('has_pre_procedure_imaging'),   has_stage('pre')),
        'has_intra_procedure_imaging': merge(c.get('has_intra_procedure_imaging'), has_stage('intra')),
        'has_post_procedure_imaging':  merge(c.get('has_post_procedure_imaging'),  has_stage('post')),
        'has_typed_report':            merge(c.get('has_typed_report'),            has_type('typed')),
        'has_handwritten_notes':       merge(c.get('has_handwritten_notes'),       has_type('handwritten')),
        'has_signed_stamp':            merge(c.get('has_signed_stamp'),            (has_type('stamp') or has_type('signature'))),
        'notes':                       c.get('notes'),
    }

def compute_derived(s, catalog):
    return {
        **s,
        'finding_correlation':   _derive_finding_correlation(s),
        'multi_image_analysis':  _derive_multi_image_analysis(catalog),
        'radiology_timeline':    _derive_radiology_timeline(catalog),
        'completeness':          _derive_completeness(s, catalog),
    }

# ── Markdown rendering — dashboard layout ───────────────────────────────
_STATUS_DOT = {'consistent': '🟢', 'partial': '🟡', 'mismatch': '🔴'}
_dash = lambda v: '—' if v in (None, '') else str(v)
_tick = lambda v: '✔' if v is True else ('✖' if v is False else '—')

def render_summary_markdown(s):
    L = []
    h = s.get('header') or {}
    st = s.get('status') or {}
    L.append(f"# Claim Summary — {s['claim_id']}"); L.append('')
    L.append(f"**Package:** {s['package']}  ·  **Model:** {s['model']}  ·  **Images:** {s['n_images']}"); L.append('')
    L.append('| Claim ID | Patient | Modality | Body Part | Study Date | Reviewer |')
    L.append('|---|---|---|---|---|---|')
    L.append(f"| {_dash(h.get('claim_id'))} | {_dash(h.get('patient_name'))} | {_dash(h.get('modality'))} | {_dash(h.get('body_part'))} | {_dash(h.get('study_date'))} | {_dash(h.get('reviewer'))} |")
    L.append('')

    dot = _STATUS_DOT.get(st.get('consistency') or '', '⚪')
    label = {'consistent': 'IMAGE & REPORT CONSISTENT', 'partial': 'PARTIAL MATCH', 'mismatch': 'MISMATCH'}.get(st.get('consistency') or '', 'STATUS UNKNOWN')
    L.append(f"> {dot} **{label}**  ·  Confidence: {st.get('confidence_pct') if st.get('confidence_pct') is not None else '—'}%  ·  Clinical Risk Score: {_dash(st.get('clinical_risk_score'))}")
    L.append('')
    if st.get('key_findings'):
        L.append('**Key findings:**')
        for f in st['key_findings']:
            ai = 'AI ✔' if f.get('ai_detected') is True else ('AI ✖' if f.get('ai_detected') is False else 'AI —')
            rp = 'Report ✔' if f.get('report_mentioned') is True else ('Report ✖' if f.get('report_mentioned') is False else 'Report —')
            note = f' — {f["note"]}' if f.get('note') else ''
            L.append(f"- {_dash(f.get('finding'))} _({ai} · {rp})_{note}")
        L.append('')

    sv = s.get('scan_viewer') or {}
    L.append('## Interactive Scan Viewer')
    L.append(f"- **Primary image:** {_dash(sv.get('primary_image_source'))}")
    L.append(f"- **AI overlays available:** {'yes' if sv.get('ai_overlays_available') else 'no'}")
    if sv.get('detected_regions'):
        L.append('- **Detected regions:**')
        for r in sv['detected_regions']:
            page = f" p.{r['page']}" if r.get('page') else ''
            L.append(f"  - {_dash(r.get('label'))} _(in {_dash(r.get('image_source'))}{page})_")
    L.append('')

    ai = s.get('ai_clinical_findings') or {}
    L.append('## AI Clinical Findings')
    def _fmt(label, x):
        if not x: L.append(f"- **{label}:** —"); return
        conf = f" ({x['confidence_pct']}%)" if x.get('confidence_pct') is not None else ''
        ev   = f" — {x['evidence']}" if x.get('evidence') else ''
        L.append(f"- **{label}:** {_tick(x.get('present'))}{conf}{ev}")
    _fmt('Fracture',           ai.get('fracture'))
    _fmt('Fluid accumulation', ai.get('fluid_accumulation'))
    _fmt('Tumor / mass',       ai.get('tumor_mass'))
    inf = ai.get('infiltration') or {}
    conf = f" ({inf['confidence_pct']}%)" if inf.get('confidence_pct') is not None else ''
    ev   = f" — {inf['evidence']}" if inf.get('evidence') else ''
    L.append(f"- **Infiltration:** {_dash(inf.get('severity'))}{conf}{ev}")
    L.append(f"- **Image quality:** {_dash(ai.get('image_quality'))}")
    L.append('')

    mi = s.get('multi_image_analysis') or {}
    L.append('## Multi-image Analysis')
    if mi.get('entries'):
        L.append('| Modality | Day | Date | Finding | Confirmed |')
        L.append('|---|---|---|---|---|')
        for e in mi['entries']:
            L.append(f"| {_dash(e.get('modality'))} | {_dash(e.get('day'))} | {_dash(e.get('date'))} | {_dash(e.get('finding'))} | {_tick(e.get('confirmed'))} |")
    else:
        L.append('_no multi-image entries_')
    L.append('')
    L.append(f"**Consistency Score:** {mi.get('consistency_score_pct') if mi.get('consistency_score_pct') is not None else '—'}%")
    L.append('')

    nlp = s.get('report_nlp_extraction') or {}
    L.append('## Report NLP Extraction')
    L.append(f"- **Reported diagnosis:** {_dash(nlp.get('reported_diagnosis'))}")
    L.append(f"- **Reported severity:** {_dash(nlp.get('reported_severity'))}")
    if nlp.get('reported_findings'):
        L.append('- **Reported findings:**')
        for x in nlp['reported_findings']: L.append(f'  - {x}')
    else:
        L.append('- **Reported findings:** —')
    L.append(f"- **Extraction confidence:** {_dash(nlp.get('extraction_confidence'))}")
    L.append('')

    fc = s.get('finding_correlation') or {}
    L.append('## Finding Correlation')
    if fc.get('rows'):
        L.append('| Finding | Image AI | Report | Match |')
        L.append('|---|---|---|---|')
        for r in fc['rows']:
            L.append(f"| {_dash(r.get('finding'))} | {_tick(r.get('image_ai'))} | {_tick(r.get('report'))} | {_tick(r.get('match'))} |")
    else:
        L.append('_no correlation rows_')
    L.append('')
    L.append(f"**Consistency Score:** {fc.get('consistency_score_pct') if fc.get('consistency_score_pct') is not None else '—'}%")
    L.append('')

    inc = s.get('inconsistency_detection') or {}
    L.append('## Inconsistency Detection')
    def _list(label, arr):
        if arr:
            L.append(f"- **{label}:**")
            for x in arr: L.append(f"  - ⚠ {x}")
        else:
            L.append(f"- **{label}:** ✔ none detected")
    _list('Possible exaggerations', inc.get('possible_exaggerations'))
    _list('Underreported findings', inc.get('underreported_findings'))
    _list('Hidden findings',        inc.get('hidden_findings'))
    L.append('')

    stg = s.get('stg_alignment') or {}
    L.append('## STG Alignment')
    L.append(f"- **Claimed package:** {_dash(stg.get('claimed_package'))}")
    if stg.get('evidence_required'):
        L.append('- **Evidence required:**')
        for e in stg['evidence_required']: L.append(f"  - {_dash(e.get('item'))} {_tick(e.get('present'))}")
    L.append(f"- **STG compliance score:** {stg.get('stg_compliance_score_pct') if stg.get('stg_compliance_score_pct') is not None else '—'}%")
    L.append('')

    tl = s.get('radiology_timeline') or {}
    L.append('## Radiology Timeline')
    if tl.get('events'):
        for e in tl['events']:
            day = f"Day {e['day']} – " if e.get('day') is not None else ''
            date = f" _({e['date']})_" if e.get('date') else ''
            L.append(f"- {day}{_dash(e.get('event'))}{date}")
    else:
        L.append('_no events_')
    L.append('')
    L.append(f"**Timeline logical:** {_tick(tl.get('logical'))}")
    L.append('')

    L.append('---'); L.append(''); L.append('## Reference detail'); L.append('')

    p = s.get('patient') or {}
    L.append('**Patient**')
    L.append(f"- Name: {_dash(p.get('name'))} · Age: {_dash(p.get('age'))} · Sex: {_dash(p.get('sex'))}")
    if p.get('id_numbers'): L.append(f"- IDs: {', '.join(p['id_numbers'])}")
    L.append('')

    ho = s.get('hospital') or {}
    L.append('**Hospital**')
    L.append(f"- Name: {_dash(ho.get('name'))} · Location: {_dash(ho.get('location'))}")
    if ho.get('doctors'): L.append(f"- Doctors: {'; '.join(ho['doctors'])}")
    L.append('')

    en = s.get('encounter') or {}
    L.append('**Encounter**')
    L.append(f"- Date range: {_dash(en.get('date_range'))} · Procedure: {_dash(en.get('primary_procedure'))} · Package code: {_dash(en.get('package_code'))}")
    if en.get('all_dates'): L.append(f"- All dates seen: {', '.join(en['all_dates'])}")
    L.append('')

    inv = s.get('image_inventory') or {}
    L.append('**Image inventory**')
    L.append(f"- Total: {inv.get('total_images') if inv.get('total_images') is not None else s['n_images']}")
    if inv.get('by_type'):
        L.append(f"- By type: {', '.join(f'{k}: {v}' for k, v in inv['by_type'].items())}")
    if inv.get('stages_present'): L.append(f"- Stages: {', '.join(inv['stages_present'])}")
    if inv.get('languages_seen'): L.append(f"- Languages: {', '.join(inv['languages_seen'])}")
    L.append('')

    if s.get('clinical_narrative'):
        L.append('**Clinical narrative**'); L.append(''); L.append(s['clinical_narrative']); L.append('')

    c = s.get('completeness') or {}
    L.append('**Completeness**')
    L.append(f"- Pre: {_tick(c.get('has_pre_procedure_imaging'))} · Intra: {_tick(c.get('has_intra_procedure_imaging'))} · Post: {_tick(c.get('has_post_procedure_imaging'))}")
    L.append(f"- Typed report: {_tick(c.get('has_typed_report'))} · Handwritten: {_tick(c.get('has_handwritten_notes'))} · Signed stamp: {_tick(c.get('has_signed_stamp'))}")
    if c.get('notes'): L.append(f"- Notes: {c['notes']}")
    L.append('')

    if s.get('concerns_or_gaps'):
        L.append('**Concerns / gaps**')
        for g in s['concerns_or_gaps']: L.append(f'- {g}')
        L.append('')

    return '\n'.join(L)

def summarize_claim(package_folder: str, claim_id: str):
    """Read image_catalog.json from the claim folder, run dashboard + reference
    calls in parallel, merge + derive, write claim_summary.{json,md}."""
    claim_dir    = CLAIMS_ROOT / package_folder / claim_id
    catalog_path = claim_dir / 'image_catalog.json'
    if not catalog_path.exists():
        print(f'  ⚠  no image_catalog.json — skip summary ({catalog_path})')
        return None

    print(f'\n→ summarise {package_folder}/{claim_id}')
    catalog = json.loads(catalog_path.read_text(encoding='utf-8'))
    compact = compact_catalog(catalog)

    user_text = (
        f'Claim: {package_folder}/{claim_id}\n'
        'Per-image catalog (JSON below). Produce the JSON object described in the system prompt. '
        'ONE JSON object. NO PROSE.\n\n'
        + json.dumps(compact, indent=2, ensure_ascii=False)
    )

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_dash = ex.submit(call_text_model, SUMMARY_SYSTEM_DASHBOARD, user_text)
        fut_ref  = ex.submit(call_text_model, SUMMARY_SYSTEM_REFERENCE, user_text)
        raw_dash = fut_dash.result()
        raw_ref  = fut_ref.result()
    dt = int((time.time() - t0) * 1000)

    merged = {**parse_json_obj(raw_ref or '{}'), **parse_json_obj(raw_dash or '{}')}
    normalized = normalize_summary(merged, package_folder, claim_id)
    obj = compute_derived(normalized, catalog)

    out = {
        'claim_id':     catalog.get('claim_id') or claim_id,
        'package':      catalog.get('package')  or package_folder,
        'model':        MODEL,
        'n_images':     catalog.get('n_images') or compact['n_images'],
        'generated_ms': dt,
        **obj,
    }

    json_path = claim_dir / 'claim_summary.json'
    json_path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'  ✓ {json_path.resolve()}  ({dt} ms)')

    md_path = claim_dir / 'claim_summary.md'
    md_path.write_text(render_summary_markdown(out), encoding='utf-8')
    print(f'  ✓ {md_path.resolve()}')
    return out

## Main pipeline — `catalog_claim` and `catalog_package`

In [10]:
def catalog_claim(package_folder: str, claim_id: str, limit: int | None = None) -> dict | None:
    claim_dir = CLAIMS_ROOT / package_folder / claim_id
    if not claim_dir.is_dir():
        raise RuntimeError(f'Not a folder: {claim_dir}')

    print(f'\n→ {package_folder}/{claim_id}')
    images = list_images_in_claim(claim_dir)
    if limit: images = images[:limit]
    if not images: print('  no images'); return None
    print(f'  {len(images)} image(s) total')

    records: list[dict] = []
    for idx, (source, page, total, img) in enumerate(images):
        label = f'{source} page {page}/{total}' if page else source
        sys.stdout.write(f'  [{idx+1}/{len(images)}] {label} … '); sys.stdout.flush()
        try:
            b64  = image_to_b64(img)
            t0   = time.time()
            resp = call_model(b64, label)
            dt   = int((time.time() - t0) * 1000)
            obj  = parse_json_obj(resp['content'] or '{}')
            rec = {
                'image_index': idx,
                'source':      source,
                'page':        page,
                'pages_total': total,
                **obj,
                '_usage': {
                    'prompt_tokens':     resp['prompt_eval_count'],
                    'completion_tokens': resp['eval_count'],
                    'total_duration_ms': dt,
                },
            }
            records.append(rec)
            print(f"{obj.get('image_type', '?')}  ({dt} ms)")
        except Exception as e:
            print(f'ERROR — {e}')
            records.append({'image_index': idx, 'source': source, 'page': page, 'error': str(e)})

    out = {
        'claim_id': claim_id,
        'package':  package_folder,
        'model':    MODEL,
        'n_images': len(records),
        'records':  records,
    }

    # Write the catalog next to the source files inside the claim folder.
    json_path = claim_dir / 'image_catalog.json'
    json_path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'  ✓ {json_path.resolve()}')

    md_path = claim_dir / 'image_catalog.md'
    md_path.write_text(render_markdown(claim_id, package_folder, records), encoding='utf-8')
    print(f'  ✓ {md_path.resolve()}')

    # Roll the per-image records up into a whole-claim summary.
    try:
        summarize_claim(package_folder, claim_id)
    except Exception as e:
        print(f'  ⚠  claim summary failed: {e}')

    return out

def catalog_package(package_folder: str, limit: int | None = None) -> None:
    pkg_dir = CLAIMS_ROOT / package_folder
    if not pkg_dir.is_dir(): raise RuntimeError(f'No folder: {pkg_dir}')
    claims = sorted([d.name for d in pkg_dir.iterdir() if d.is_dir()])
    for c in claims:
        try: catalog_claim(package_folder, c, limit=limit)
        except Exception as e: print(f'  ✗ {c}: {e}')

## Run — single claim

Edit `PACKAGE` / `CLAIM` and run the cell. Set `limit=3` while iterating to keep cost low.

In [13]:
PACKAGE = 'MG029A'
CLAIM   = 'MAV_GJ_R3_2026033110059264'

result = catalog_claim(PACKAGE, CLAIM)  # add limit=3 to test cheap


→ MG029A/MAV_GJ_R3_2026033110059264
  2 image(s) total
  [1/2] 000021__MAV_GJ_R3_2026033110059264__X_RAY_REPORT.pdf page 1/1 … Typed report  (4320 ms)
  [2/2] 000063__MAV_GJ_R3_2026033110059264__X_RAY_PLATE.jpeg … Chest X-Ray  (2945 ms)
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\image_catalog.json
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\image_catalog.md

→ summarise MG029A/MAV_GJ_R3_2026033110059264
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\claim_summary.md  (3571 ms)


### Inspect a single record without scrolling the JSON

In [12]:
if result and result['records']:
    r = result['records'][0]
    print(f"Image #{r['image_index']} — {r['source']}" + (f" page {r['page']}/{r['pages_total']}" if r.get('page') else ''))
    print(f"Type: {r.get('image_type')} · Body: {r.get('body_region')} · Stage: {r.get('stage_of_care')}")
    print('Summary:', r.get('summary'))
    print('Dates:',   r.get('dates_found'))

Image #0 — 000019__MMJAA_UP_2026_R3_2026033010053802__MALATIKUV.pdf page 1/2
Type: Typed report · Body: Abdomen — pelvicalyceal system · Stage: n/a
Summary: This is a typed radiology report for an X-ray KUB (Kidney, Ureter, Bladder) performed on a 36-year-old female. The report notes the absence of significant radioopacity in the renal fossa, normal appearance of surrounding soft tissues and fat shadows, and confirms the presence of a DJ stent in the left side with its ends positioned in the renal region and pelvis, respectively. The impression is 'Normal Study' with advice for clinical correlation.
Dates: ['02/04/2026']


## Run — every claim in a package

Each iteration writes its own `image_catalog.json` and `.md` inside that claim's folder.

In [13]:
# catalog_package('SU007A')
# catalog_package('MC011A')
# catalog_package('MG029A')
# catalog_package('SG039')   # alias for SG039C — folder is named SG039 in this repo

## Switch to local Ollama mid-session

Re-run the **Config** cell after changing the env var.

In [14]:
# os.environ['PROVIDER'] = 'ollama'
# os.environ.pop('MODEL', None)   # use the Ollama default (qwen3-vl:8b-instruct)
# # then re-run the config cell + run cell